In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print("Project root:", project_root)

In [ ]:
import time
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

from app.weather_fetch import (
    generate_weather_points,
    assign_grid_to_weather_points,
    fetch_all_weather_points,
    add_antecedent_precip,
)

## 1. Load boundary and 1km grid

In [ ]:
boundary = gpd.read_file("../data/raw/boundaries/kamrup_metropolitan.geojson")
grid = gpd.read_parquet("../data/processed/kamrup_metro_grid_1km.parquet")

print("Boundary rows:", len(boundary))
print("Grid cells:", len(grid))

## 2. Generate coarse weather sample points (~9km spacing)

Open-Meteo's ERA5-Land archive is natively ~9km resolution, far coarser than our 1km analysis grid. Fetching weather per grid cell would issue ~40x more API calls than necessary and would misrepresent 9km data as 1km-resolution weather. Instead we generate a coarse set of sample points spaced ~9km apart over the district's bounding box, fetch each once, and assign every 1km grid cell to its nearest sample point (nearest-neighbour downscaling).

In [ ]:
weather_points = generate_weather_points(boundary, spacing_km=9)
print("Weather sample points generated (covering bounding box):", len(weather_points))
weather_points[["weather_point_id", "lat", "lon"]]

## 3. Assign each grid cell to its nearest weather point

In [ ]:
mapping = assign_grid_to_weather_points(grid, weather_points)

used_point_ids = mapping["weather_point_id"].unique()
weather_points_used = weather_points[weather_points["weather_point_id"].isin(used_point_ids)].reset_index(drop=True)

print("Grid cells mapped:", len(mapping))
print("Weather points actually used by at least one grid cell:", len(weather_points_used))
print("Weather points generated but unused (outside district shape):", len(weather_points) - len(weather_points_used))
print("Max grid-cell-to-weather-point distance (m):", mapping["distance_m"].max())
mapping["weather_point_id"].value_counts().describe()

## 4. Fetch hourly weather (cached — re-running this cell will not re-download data that's already on disk)

Only the weather points actually used by the grid (step 3) are fetched — fetching the unused bbox-corner points would be pure waste, exactly what this design is trying to avoid.

In [ ]:
START_DATE = "2018-01-01"
END_DATE = "2025-12-31"
CACHE_DIR = "../data/raw/weather_cache"

weather_df, stats = fetch_all_weather_points(weather_points_used, START_DATE, END_DATE, CACHE_DIR)

print("Weather points fetched:", stats["n_points"])
print("Actual API calls made (not cached):", stats["n_api_calls"])
print("Served from cache:", stats["n_cache_hits"])
print("Total rows fetched:", stats["total_rows"])
print(f"Wall-clock time: {stats['wall_time_seconds']:.1f} s")

## 5. Antecedent Precipitation Index (api_3d, api_7d)

Computed per weather_point_id, on this ~9km table, **before** the grid join in step 3 — each weather point is shared by ~40+ grid cells, so computing the rolling sum after joining onto grid_id would recompute identical values that many times over for no benefit.

In [ ]:
weather_df = add_antecedent_precip(weather_df, windows_days=(3, 7))
print("Columns after adding API:", list(weather_df.columns))
weather_df[["weather_point_id", "timestamp", "precipitation_mm", "api_3d", "api_7d"]].head()

## 6. Save outputs

In [ ]:
weather_out_path = "../data/processed/weather_hourly.parquet"
mapping_out_path = "../data/processed/grid_weather_mapping.parquet"

weather_df.to_parquet(weather_out_path)
mapping.to_parquet(mapping_out_path)

import os
size_mb = os.path.getsize(weather_out_path) / 1e6
print("Saved:", weather_out_path, f"({size_mb:.2f} MB)")
print("Saved:", mapping_out_path)
print("weather_df columns:", list(weather_df.columns))
print("weather_df dtypes:")
print(weather_df.dtypes)

## 7. Sanity-check plot — monthly rainfall for one weather point

Raw hourly precipitation over 8 years is unreadable directly, so we sum to monthly totals for one representative point and look for the expected Guwahati monsoon pattern (a strong Jun-Sep peak each year).

In [ ]:
sample_point_id = mapping["weather_point_id"].value_counts().idxmax()
print("Plotting point (most grid cells assigned to it):", sample_point_id)

point_df = weather_df[weather_df["weather_point_id"] == sample_point_id].copy()
point_df = point_df.set_index("timestamp")
monthly_precip = point_df["precipitation_mm"].resample("MS").sum()

fig, ax = plt.subplots(figsize=(14, 5))
monthly_precip.plot(ax=ax, color="steelblue")
ax.set_title(f"Monthly total precipitation - {sample_point_id} (2018-2025)")
ax.set_xlabel("Month")
ax.set_ylabel("Precipitation (mm)")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../docs/img/03_weather_monthly_precip.png", dpi=150)
plt.show()